# How much does one production release cost?

`chunk_storage_stats()` reports storage for a whole icechunk repository. The production store keeps
one branch per release (`v0.12.0`, `v0.13.0`, ...) beside `main`, and there is no per-branch size
API, so that one number can't be split by inspection. This notebook splits it. Point it at a store,
run it, read the table.

Nothing here is destructive and nothing reads array data — only manifests and zarr metadata.

**Method.** Manifests record chunk *counts*, never bytes. So: read each array's logical size from
zarr metadata (`prod(shape) * itemsize`), divide by its chunk-ref count to get bytes-per-ref (the
arrays are sharded, so one ref is one shard object), then walk each branch's full ancestry and union
the manifest ids it reaches. That union counts superseded shards left behind by intermediate
commits, not just what survives at the tip. Multiply the total by
`native_bytes / total_logical_bytes` to convert logical bytes to stored bytes.

The ratio is calibrated store-wide, so branch numbers sum back to the reported total by
construction. The *split* is the result, not the total.

In [ ]:
bucket = "us-west-2.opendata.source.coop"
prefix = "carbonplan/srm-downscaling/output/production/CESM2-WACCM-ERA5-global.icechunk"
region = "us-west-2"
anonymous = True  # the production store is public; set False to use your own credentials
branches = None  # None -> every branch in the repo
run_gc_dry_run = False  # see the last section; needs write credentials, deletes nothing

In [ ]:
import icechunk
import numpy as np
import pandas as pd
import zarr

from saidownscale.config import _icechunk_storage_for_path

GB = 1e9
TB = 1e12

storage = (
    icechunk.s3_storage(bucket=bucket, prefix=prefix, anonymous=True, region=region)
    if anonymous
    else _icechunk_storage_for_path(f"s3://{bucket}/{prefix}")
)
repo = icechunk.Repository.open(storage)

refs = sorted(branches or repo.list_branches())
ancestry = {b: [s.id for s in repo.ancestry(branch=b)] for b in refs}
{b: len(ids) for b, ids in ancestry.items()}

## Measure

Two object fetches per snapshot, so expect one to three minutes on a full production store.

In [ ]:
def snapshot_arrays(snapshot_id: str) -> dict[str, dict]:
    """path -> {manifests: {id: refs}, refs} for one snapshot, from metadata only."""
    counts = {m.id: m.num_chunk_refs for m in repo.list_manifest_files(snapshot_id)}
    out = {}
    for node in repo.inspect_snapshot(snapshot_id)["nodes"]:
        if node["node_type"] != "array":
            continue
        manifests = {r["id"]: counts.get(r["id"], 0) for r in node.get("manifest_refs") or []}
        out[node["path"]] = {"manifests": manifests, "refs": sum(manifests.values())}
    return out


def branch_stats(branch: str) -> dict:
    """Logical bytes and chunk refs for a branch, at its tip and over its full ancestry."""
    session = repo.readonly_session(branch=branch)
    root = zarr.open_group(session.store, mode="r", zarr_format=3)

    per_ref, tip_logical, tip_refs = {}, 0.0, 0
    for path, info in snapshot_arrays(repo.lookup_branch(branch)).items():
        arr = root[path.lstrip("/")]
        logical = int(np.prod(arr.shape)) * arr.dtype.itemsize
        tip_logical += logical
        tip_refs += info["refs"]
        if info["refs"]:
            per_ref[path] = logical / info["refs"]

    manifests: dict[str, tuple[str, int]] = {}
    for snapshot_id in ancestry[branch]:
        for path, info in snapshot_arrays(snapshot_id).items():
            for mid, n_refs in info["manifests"].items():
                manifests.setdefault(mid, (path, n_refs))

    # paths that only ever existed in expired history fall back to the median shard size
    fallback = float(np.median(list(per_ref.values()))) if per_ref else 0.0
    union_logical = sum(n * per_ref.get(path, fallback) for path, n in manifests.values())

    return {
        "branch": branch,
        "snapshots": len(ancestry[branch]),
        "tip_refs": tip_refs,
        "history_refs": sum(n for _, n in manifests.values()),
        "tip_logical": tip_logical,
        "history_logical": union_logical,
    }


stats = repo.chunk_storage_stats()
native_bytes = stats.native_bytes
sizes = pd.DataFrame([branch_stats(b) for b in refs]).set_index("branch")

compression_ratio = native_bytes / sizes.history_logical.sum()
assert compression_ratio <= 1.0, f"ratio {compression_ratio:.3f} > 1: logical bytes undercounted"
print(f"compression ratio {compression_ratio:.3f} ({1 / compression_ratio:.2f}x)")

## Size per branch

`live_TB` is what the tip still points at. `stored_TB` is what the branch actually occupies,
including shards that intermediate commits superseded but never released.

In [ ]:
sizes["stored_TB"] = sizes.history_logical * compression_ratio / TB
sizes["live_TB"] = sizes.tip_logical * compression_ratio / TB
sizes["share_pct"] = 100 * sizes.stored_TB / sizes.stored_TB.sum()

print(f"repository total: {native_bytes / TB:.2f} TB native")
print(f"branches sum to:  {sizes.stored_TB.sum():.2f} TB")
sizes[["snapshots", "live_TB", "stored_TB", "share_pct"]].round(3)

## Reclaimable by expiring intermediate snapshots

Every commit that rewrote a region left its old shards behind. They are still stored and still
billed, and garbage collection alone will not touch them — the intermediate snapshots that reference
them are reachable from a branch, so by icechunk's definition the shards are live.

`expire_snapshots` is what drops those intermediate snapshots. With the default flags it never
expires the root snapshot, the `main` tip, or any other branch or tag tip, so every release stays
readable at its published version; what disappears is the per-commit history *inside* each release.
A `garbage_collect` afterwards is what actually reclaims the bytes.

The table below is the size of that prize. Nothing below runs either operation.

In [ ]:
waste = pd.DataFrame(
    {
        "dead_refs": sizes.history_refs - sizes.tip_refs,
        "reclaimable_TB": (sizes.history_logical - sizes.tip_logical) * compression_ratio / TB,
    }
)
waste["pct_of_branch"] = 100 * waste.reclaimable_TB / sizes.stored_TB.replace(0, np.nan)

print(
    f"total reclaimable: {waste.reclaimable_TB.sum():.2f} TB "
    f"({100 * waste.reclaimable_TB.sum() / (native_bytes / TB):.1f}% of the store)"
)
waste.round(3)

## Cost per production unit

One unit is one `(scenario, variable, member)` triple. Output paths are
`/{scenario}/{variable}/{member}/{variable}`, with the coarse intermediates under
`/debiased_coarse/...`; the two tiers differ by more than an order of magnitude, so they are kept
apart. Costs come from the newest branch, since older releases carry older array geometry.

In [ ]:
newest = max(refs, key=lambda b: next(repo.ancestry(branch=b)).written_at)
session = repo.readonly_session(branch=newest)
root = zarr.open_group(session.store, mode="r", zarr_format=3)

rows = []
for path, info in snapshot_arrays(repo.lookup_branch(newest)).items():
    parts = path.strip("/").split("/")
    tier = "fine"
    if parts and parts[0] == "debiased_coarse":
        tier, parts = "coarse", parts[1:]
    if len(parts) < 3:
        continue
    arr = root[path.lstrip("/")]
    rows.append(
        {
            "tier": tier,
            "unit": "/".join(parts[:3]),
            "stored_GB": int(np.prod(arr.shape)) * arr.dtype.itemsize * compression_ratio / GB,
            "years": arr.shape[0] / 365.25 if arr.ndim == 3 else np.nan,
        }
    )

units = (
    pd.DataFrame(rows)
    .groupby(["tier", "unit"])
    .agg(stored_GB=("stored_GB", "sum"), years=("years", "max"))
)
units["GB_per_year"] = units.stored_GB / units.years

unit_cost = units.groupby("tier").agg(
    n_units=("stored_GB", "size"),
    median_GB=("stored_GB", "median"),
    median_GB_per_year=("GB_per_year", "median"),
)
print(f"unit costs from {newest}")
unit_cost.round(3)

In [ ]:
per_unit_GB = unit_cost.median_GB.sum()  # one fine unit plus the coarse intermediate it implies
print(f"one additional (scenario, variable, member): {per_unit_GB:.1f} GB\n")
for n in [1, 5, 10, 25, 50]:
    print(f"{n:3d} additional units -> {n * per_unit_GB / 1000:6.2f} TB")

## Optional: orphaned objects

`chunk_storage_stats()` counts only what a branch or tag can reach. Objects orphaned by an earlier
`reset_branch` or expiration still sit in the bucket and are still billed.
`garbage_collect(..., dry_run=True)` measures them without deleting anything.

**Needs write credentials** (set `anonymous=False`), and it is off by default. `dry_run=True` deletes
nothing — but the same call with `dry_run=False` is irreversible, so leave that argument alone.

In [ ]:
if run_gc_dry_run:
    from datetime import UTC, datetime

    summary = repo.garbage_collect(datetime.now(UTC), dry_run=True)
    print(
        f"orphaned: {summary.bytes_deleted / TB:.3f} TB across {summary.chunks_deleted} chunks, "
        f"{summary.manifests_deleted} manifests, {summary.snapshots_deleted} snapshots "
        f"(nothing was deleted)"
    )
else:
    print("skipped; set run_gc_dry_run=True with write credentials to measure orphaned objects")